# 08 · v2 re-run: A, D, C, C-hedge, B

用 v2 holdings-based simulator 重跑已经收尾的 A（carry）、D（curve momentum）和
C（mean reversion 1:1），并把 C 的 rolling-OLS / Kalman hedge-ratio 变体和
B（adjacent-carry pre-test）接进 v2。所有口径见 D-018；v1 数字只作对照，
不和 v2 混在同一张结论表里。

## 1. 口径和注意事项

- **v2 default**：`d=5`、`k=5`、`min_adv=100`、equal-notional legs、
  three-layer costs（half-spread + USD 1.5 fee + square-root impact）、
  `resize="on_change"`、capital USD 10m。
- **calendar 策略**：C/D/B 传一个很大的 `rebalance_band`，让只有 target /
  composition 变化才调仓；A 保留 v2 默认的 0.25 band。
- **C hedge/Kalman**：h 按周 snap；pair-change 日使用当天 h；invalid h 平仓。
- **B**：保留 v1 的 raw-rank adjacent universe（不是 v2 的 `d=5` universe），
  `execution="legs"`，按 v1 的 weekly 日期加 main-pair change 日期采样。
- **v1 vs v2 不可直接比较**：`d=None` vs `d=5`、1:1 contracts vs
  equal-notional、5 bps/leg vs 三层成本、gross-notional vs capital、
  geometric vs arithmetic 年化、turnover 定义不同、test 是 post-hoc slice。
- **`d=5` 不是 point-in-time**：`curve.add_contract_calendar` 用
  `last_trade_proxy`，这里如实报告，不把它当作 ex-ante clean。
- **负价格**：v2 只交易正价格，CL 2020-04-20 不可交易；相关日期会被跳过。
- **flat-day 修复**：v2 的 daily return 现在对齐完整 calendar 并零填平仓日；
  因此本次数字和 notebook 07 里 merge 时带进来的旧输出可能不同。
- 不做 test-set tuning；test 结果只用于报告。

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
OUT = ROOT / "output" / "v2_rerun"

if not (OUT / "comparison.csv").exists():
    from src import v2_rerun
    rank = pd.read_parquet(ROOT / "data" / "processed" / "rank.parquet")
    v2_rerun.save_results(v2_rerun.run_all(v2_rerun.STRATEGIES, rank), OUT)

comparison = pd.read_csv(OUT / "comparison.csv")


def load_perf(name: str) -> pd.DataFrame:
    return pd.read_csv(OUT / f"{name}_performance.csv", index_col=[0, 1])


## 2. 汇总：full-sample 四层收益

`gross` 是 v2 的 gross return；后面三层依次扣 half-spread、fee、impact。
v1 的 net 是固定 5 bps/leg，不能和 `- impact (all)` 直接对齐。

In [2]:
layers = ["gross", "- half-spread", "- fees", "- impact (all)"]
full = comparison.loc[comparison["period"].eq("full")]
table = (
    full.pivot(index="strategy", columns="layer", values="ann_ret_pct")
    .reindex(columns=layers)
)
display(table.round(3))


layer,gross,- half-spread,- fees,- impact (all)
strategy,,,,
a,-0.484,-0.843,-1.002,-2.610
b,-1.131,-2.180,-2.374,-4.469
c,0.614,0.027,-0.226,-3.695
c_kalman,0.613,-0.175,-0.497,-6.099
c_ols,0.124,-0.449,-0.693,-3.060
d,0.219,-0.074,-0.202,-1.418


In [3]:
selected = comparison.loc[comparison["layer"].isin(["gross", "- impact (all)"])]
display(
    selected.pivot(index=["strategy", "layer"], columns="period", values="ann_ret_pct")
    .round(3)
)


period                    full  test 2016-2026  train 1986-2015
strategy layer                                                 
a        - impact (all) -2.610          -1.464           -3.024
         gross          -0.484          -0.272           -0.560
b        - impact (all) -4.469          -3.643           -4.767
         gross          -1.131          -1.229           -1.095
c        - impact (all) -3.695          -2.059           -4.285
         gross           0.614           0.164            0.776
c_kalman - impact (all) -6.099          -3.857           -6.907
         gross           0.613           0.982            0.479
c_ols    - impact (all) -3.060          -1.838           -3.501
         gross           0.124          -0.010            0.173
d        - impact (all) -1.418          -1.084           -1.539
         gross           0.219          -0.205            0.372

**读法：** 六个策略在 v2 全成本下都是负的。A/B 的 gross 已经为负；
D/C/C-OLS/C-Kalman 的 gross 为正，但 turnover 高、impact 把 net 拖到负。
D 和 B 的 test gross 也为负；C 和 C-Kalman 的 test gross 为正，但成本后仍为负。
这不是 v2 算错，而是 v2 把 v1 的 5 bps 拆成了三层，并且 universe/pair rule
不同。

## 3. A — carry

v2-native：daily target、equal-notional、默认 0.25 band。结果和 notebook 07
的 A 示例一致（gross ≈ −0.48%/yr）。

In [4]:
display(load_perf("a").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross       -0.484        0.957  -0.506  -3.456   
                1   - half-spread       -0.843        0.958  -0.880  -5.988   
                2          - fees       -1.002        0.959  -1.045  -7.098   
                3  - impact (all)       -2.610        1.006  -2.595 -16.528   
train 1986-2015 0           gross       -0.560        0.995  -0.563  -3.437   
                1   - half-spread       -0.973        0.996  -0.977  -5.942   
                2          - fees       -1.154        0.997  -1.157  -7.020   
                3  - impact (all)       -3.024        1.053  -2.871 -16.107   
test 2016-2026  0           gross       -0.272        0.843  -0.323  -0.997   
                1   - half-spread       -0.480        0.843  -0.569  -1.752   
                2          - fees       -0.580        0.844  -0.688  -2.115   
                3  - impact (all)       -1.464        0.856  -1.710  -5.172   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0     -24.239      26.133        0.000  
                1     -38.224      26.133       35.873  
                2     -44.462      26.133       51.806  
                3    -109.099      26.133      212.639  
train 1986-2015 0     -18.063      26.644        0.000  
                1     -30.556      26.644       41.333  
                2     -36.022      26.644       59.398  
                3     -92.712      26.644      246.368  
test 2016-2026  0      -6.220      24.718        0.000  
                1      -7.646      24.718       20.742  
                2      -8.419      24.718       30.767  
                3     -16.361      24.718      119.166

## 4. D — curve momentum

D 的 target 是月度 forward-filled path；v2 在 target 或 pair 变化时调仓，
并用大 band 抑制预算漂移。v1 的 D 是 monthly rebalance + 1:1 contracts +
5 bps；v2 是 `d=5` + equal-notional + 三层成本，所以 gross 也不同。

In [5]:
display(load_perf("d").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross        0.219        0.881   0.248   1.767   
                1   - half-spread       -0.074        0.883  -0.084  -0.597   
                2          - fees       -0.202        0.884  -0.229  -1.623   
                3  - impact (all)       -1.418        0.931  -1.523 -10.426   
train 1986-2015 0           gross        0.372        0.909   0.409   2.604   
                1   - half-spread        0.036        0.911   0.039   0.248   
                2          - fees       -0.108        0.913  -0.119  -0.751   
                3  - impact (all)       -1.539        0.970  -1.586  -9.552   
test 2016-2026  0           gross       -0.205        0.800  -0.256  -0.827   
                1   - half-spread       -0.379        0.800  -0.473  -1.529   
                2          - fees       -0.464        0.800  -0.579  -1.871   
                3  - impact (all)       -1.084        0.813  -1.334  -4.302   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0      -5.510      22.078        0.000  
                1      -7.338      22.078       29.323  
                2     -11.375      22.078       42.148  
                3     -60.255      22.078      163.715  
train 1986-2015 0      -2.741      21.878        0.000  
                1      -4.460      21.878       33.630  
                2      -7.201      21.878       48.022  
                3     -47.326      21.878      191.053  
test 2016-2026  0      -4.959      22.634        0.000  
                1      -6.152      22.634       17.384  
                2      -6.981      22.634       25.868  
                3     -12.994      22.634       87.952

## 5. C — mean reversion 1:1

C 的 target 每周更新；v2 用 equal-notional legs。`min_adv=100` 的不流动日期
是显式平仓，因此 turnover 比 v1 的 1:1 baseline 高。

In [6]:
display(load_perf("c").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross        0.614        1.634   0.376   2.619   
                1   - half-spread        0.027        1.634   0.017   0.117   
                2          - fees       -0.226        1.635  -0.138  -0.966   
                3  - impact (all)       -3.695        1.795  -2.059 -14.227   
train 1986-2015 0           gross        0.776        1.698   0.457   2.802   
                1   - half-spread        0.104        1.699   0.061   0.375   
                2          - fees       -0.179        1.701  -0.105  -0.649   
                3  - impact (all)       -4.285        1.900  -2.255 -13.633   
test 2016-2026  0           gross        0.164        1.441   0.114   0.375   
                1   - half-spread       -0.184        1.439  -0.128  -0.420   
                2          - fees       -0.354        1.438  -0.246  -0.812   
                3  - impact (all)       -2.059        1.457  -1.414  -4.694   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0      -5.408      43.170        0.000  
                1      -9.212      43.170       58.669  
                2     -14.396      43.170       83.986  
                3    -154.522      43.170      430.925  
train 1986-2015 0      -4.623      42.558        0.000  
                1      -6.081      42.558       67.291  
                2      -9.624      42.558       95.577  
                3    -132.064      42.558      506.183  
test 2016-2026  0      -5.408      44.867        0.000  
                1      -6.791      44.867       34.772  
                2      -7.488      44.867       51.862  
                3     -23.374      44.867      222.359

## 6. C — rolling-OLS hedge ratio

60-day rolling OLS levels；h 在 weekly evaluation 日 snap，pair-change 日使用
当天 h；invalid h 平仓。far units = `-sign(target) * h`。

In [7]:
display(load_perf("c_ols").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross        0.124        1.169   0.106   0.688   
                1   - half-spread       -0.449        1.172  -0.383  -2.466   
                2          - fees       -0.693        1.174  -0.590  -3.801   
                3  - impact (all)       -3.060        1.266  -2.417 -15.022   
train 1986-2015 0           gross        0.173        1.207   0.143   0.802   
                1   - half-spread       -0.478        1.211  -0.395  -2.200   
                2          - fees       -0.751        1.213  -0.619  -3.445   
                3  - impact (all)       -3.501        1.325  -2.642 -14.087   
test 2016-2026  0           gross       -0.010        1.058  -0.009  -0.030   
                1   - half-spread       -0.367        1.059  -0.346  -1.116   
                2          - fees       -0.533        1.060  -0.503  -1.623   
                3  - impact (all)       -1.838        1.082  -1.699  -5.510   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0      -5.150      42.738        0.000  
                1     -19.207      42.738       57.283  
                2     -29.330      42.738       81.776  
                3    -127.761      42.738      318.458  
train 1986-2015 0      -5.150      42.244        0.000  
                1     -15.638      42.244       65.086  
                2     -23.605      42.244       92.409  
                3    -107.730      42.244      367.395  
test 2016-2026  0      -3.893      44.108        0.000  
                1      -5.787      44.108       35.660  
                2      -6.715      44.108       52.307  
                3     -20.636      44.108      182.835

## 7. C — Kalman hedge ratio

paper-style DLM（252-day likelihood、每 60 天重估 V/W、9 点 q grid）；同样的
weekly h snap 和 invalid-h flat。Kalman 的 turnover 最高，impact 也最大。

In [8]:
display(load_perf("c_kalman").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross        0.613        2.446   0.250   1.624   
                1   - half-spread       -0.175        2.447  -0.072  -0.465   
                2          - fees       -0.497        2.448  -0.203  -1.318   
                3  - impact (all)       -6.099        2.724  -2.239 -14.501   
train 1986-2015 0           gross        0.479        2.285   0.210   1.131   
                1   - half-spread       -0.399        2.288  -0.174  -0.942   
                2          - fees       -0.751        2.291  -0.328  -1.774   
                3  - impact (all)       -6.907        2.656  -2.601 -14.077   
test 2016-2026  0           gross        0.982        2.844   0.345   1.224   
                1   - half-spread        0.445        2.840   0.157   0.556   
                2          - fees        0.209        2.839   0.074   0.261   
                3  - impact (all)       -3.857        2.901  -1.330  -4.778   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0     -11.014      54.716        0.000  
                1     -15.498      54.716       78.776  
                2     -25.800      54.716      110.916  
                3    -254.559      54.716      671.106  
train 1986-2015 0     -11.014      53.649        0.000  
                1     -15.498      53.649       87.813  
                2     -25.800      53.649      123.022  
                3    -212.026      53.649      738.638  
test 2016-2026  0      -7.104      57.673        0.000  
                1      -8.041      57.673       53.729  
                2      -8.449      57.673       77.366  
                3     -45.114      57.673      483.950

## 8. B — adjacent-carry pre-test

B 保留 v1 的 raw-rank adjacent carries 和 `d=None` main-pair 验证；v2 执行时
用 `execution="legs"`（因为 listed-spread cost pairing 不是 group-aware），
并按 weekly + main-pair change 日期采样。B 是 D-017 的 within-root
cross-sectional exception。

In [9]:
display(load_perf("b").round(3))

layer  ann_ret_pct  ann_vol_pct  sharpe    t_nw  \
period                                                                        
full            0           gross       -1.131        0.895  -1.263  -9.521   
                1   - half-spread       -2.180        0.904  -2.410 -18.017   
                2          - fees       -2.374        0.908  -2.616 -19.519   
                3  - impact (all)       -4.469        0.988  -4.525 -32.954   
train 1986-2015 0           gross       -1.095        0.753  -1.454  -8.486   
                1   - half-spread       -2.302        0.768  -2.999 -17.419   
                2          - fees       -2.523        0.773  -3.265 -18.961   
                3  - impact (all)       -4.767        0.881  -5.412 -31.277   
test 2016-2026  0           gross       -1.229        1.204  -1.020  -4.558   
                1   - half-spread       -1.840        1.204  -1.528  -6.777   
                2          - fees       -1.961        1.205  -1.628  -7.211   
                3  - impact (all)       -3.643        1.235  -2.950 -12.784   

                   max_dd_pct  turnover_x  cost_bps_yr  
period                                                  
full            0     -48.872      31.017        0.000  
                1     -91.638      31.017      104.890  
                2     -99.608      31.017      124.358  
                3    -186.717      31.017      333.822  
train 1986-2015 0     -35.283      31.438        0.000  
                1     -70.736      31.438      120.681  
                2     -77.376      31.438      142.796  
                3    -146.050      31.438      367.146  
test 2016-2026  0     -14.839      29.848        0.000  
                1     -21.158      29.848       61.128  
                2     -22.408      29.848       73.260  
                3     -40.797      29.848      241.469

## 9. v1 vs v2 框架对照

| 维度 | v1 | v2 |
|---|---|---|
| 账户 | 无 capital，gross-notional return | capital-based，fractional contracts |
| 成本 | 固定 5 bps/leg | half-spread + USD 1.5 fee + sqrt impact |
| 调仓 | weekly / monthly 日历 | `on_change` 或 daily；target/composition 驱动 |
| 腿比例 | 1:1 contracts（可带 hedge ratio） | 默认 equal-notional；可选 hedge ratio |
| 配对 | `d=None` + K=5 | `d=5` + K=5 + `min_adv=100` |
| 诊断 | summary / sector / conditional mean | horizon decay / IC / bootstrap / k sweep / capacity |
| 多腿 | `build_target_legs` 任意腿 | `legs_from_rows`；B 需要显式多腿 |
| 缺失/平仓 | 最后有效 mark，force flat 有延迟 | tradable flag，显式 flat，按日检查 |
| 年化 | geometric（cumprod） | arithmetic（mean × 252，零填平仓日） |
| test | 从 2016 空仓重跑 | 全样本 post-hoc slice |

v1 适合做 A 的原始 benchmark；v2 适合成本归因、horizon 和容量分析。
两者不是同一套会计，不能把 v1 net 和 v2 net 当成同一个数字。

## 10. 结论

1. **A/B 仍然是负结果**：v2 gross 分别为 −0.48% / −1.13%，全成本后更差。
2. **D/C/C-OLS/C-Kalman 的 gross 为正但很弱**：0.12%–0.61%/yr；impact 层
   把所有 net 拖到 −1.4% 到 −6.1%/yr。
3. **成本结构比 v1 的 5 bps 更真实**：C/C-Kalman/B 的高 turnover 让 impact
   成为主成本；这不是参数调优能解决的。
4. **D 的 test gross 为负、B 的 full/test gross 都为负**；C 和 C-Kalman 的
   test gross 为正，但 net 仍负。结论仍是 negative/mixed，不做 rescue。
5. **框架差异**：本次是 v2-native 口径，不是 v1 复现；`d=5` 还有 ex-post
   proxy 的 caveat，test 是 post-hoc slice。报告里必须写清楚。
6. 不 push；下一步是 E（Nelson–Siegel）和 H（holding-period scan）。